### Tools

Models can request to call that perform tasks such as searching the web, accessing a database, or calling an API. 
This allows the model to provide more accurate and up-to-date information, as well as perform actions on behalf of the user.

1. A schema, including the name of the tool, a description of what it does, and the parameters it accepts.
2. A function that implements the tool's functionality.

In [78]:
import os
from langchain.chat_models import init_chat_model
import dotenv
dotenv.load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

response = model.stream("Essay on Ai")

for chunk in response:    print(chunk.text, end="")

<think>
Okay, I need to write an essay on AI. Let me start by understanding what the user is looking for. They probably want a comprehensive overview of AI, covering its history, current state, applications, challenges, and future implications. Let me break this down.

First, I should define artificial intelligence. Maybe start with the basic concept and then move to its branches like machine learning, deep learning, natural language processing, etc. It's important to mention key milestones in AI history, like the Dartmouth Conference in 1956, which is often considered the birth of AI. Then talk about major developments over time, such as expert systems in the 1980s, the rise of machine learning in the 2000s with big data, and recent advancements in deep learning.

Next, I need to cover applications. AI is everywhere now—healthcare, finance, autonomous vehicles, customer service, etc. Maybe give specific examples like IBM Watson in healthcare or self-driving cars by Tesla. Also, touch 

In [79]:
# Tools

from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather information in a given location"""  # This is description of the tool is important for the agent to understand how to use it.

    # This is a mock implementation. In a real implementation, you would call a weather API to get the actual weather data.
    return f"The current weather in {location} is sunny." 


model_with_tools = model.bind_tools([get_weather])

In [80]:
response = model_with_tools.invoke("What is the current weather in Tehran?")

print(response)

for tool_call in response.tool_calls:
    print(f"Tool call: {tool_call['name']}")
    print(f"Args: {tool_call['args']}\n")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the current weather in Tehran. I need to use the get_weather function. The function requires a location parameter. Tehran is the location here, so I should call the function with "Tehran" as the argument. Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': '5rrqbteb7', 'function': {'arguments': '{"location":"Tehran"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 157, 'total_tokens': 257, 'completion_time': 0.165187167, 'completion_tokens_details': {'reasoning_tokens': 74}, 'prompt_time': 0.006402909, 'prompt_tokens_details': None, 'queue_time': 0.04701101, 'total_time': 0.171590076}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'

In [81]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Tehran'},
  'id': '5rrqbteb7',
  'type': 'tool_call'}]

### Tool Execution Loop

#### Important thing which I faced: 

- Check which model are you using. If the model you are using does not provide any " reasoning_content in the additional_kwags", then
the model cannot produce the output you are looking for. 

- For example, right before I was using lLAMA-8b-instant, which was not giving me the proper output. When I switch the model to Qwen, which will be providing reasoning_content in the AI message, I got the correct answer. 

- It is important: which model are you using? 

In [83]:
# Step 1 : Model generates a response using the tool
message = [{"role": "user", "content": "What is the weather in Tehran?"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)

# Step 2 : Execute the tool calls and collect the results
for tool_call in ai_msg.tool_calls:
    # Execute the tool call with the provided arguments and get the result
    result = get_weather.invoke(tool_call)
    message.append(result)

# Step 3 : Model generates a final response using the tool results
final_response = model_with_tools.invoke(message)
print(final_response.content)

The current weather in Tehran is sunny. It's a great day to enjoy outdoor activities! 😊
